# GenEuclidean v3.0 — End-to-End 3D Pocket-to-LLM Drug Design & Benchmark Suite (Google Colab GPU)

**SE(3)-Equivariant 3D Protein Pocket Conditioning (`PaiNN-SE3` / `e3nn` / `EGNN` / `SchNet`) $\rightarrow$ Permutation-Invariant `PocketPerceiverResampler` + `PocketCVAE` $\rightarrow$ Pretrained Chemical Foundation Transformers (`GP-MoLFormer-Uniq` 1.1B SMILES, `GPT2-ZINC-87M`) & Local LLMs (`Qwen2.5-0.5B/1.5B`, `Qwen3-0.6B-Base`, `SmolLM2-360M`) + LoRA RPO-DPO Alignment + Native Linux AutoDock Vina v1.2.7 + PoseBusters.**

### How to Use This Notebook
1. **Runtime:** Choose **Runtime $\rightarrow$ Change runtime type $\rightarrow$ GPU** (`A100` 40GB recommended for multi-architecture + `e3nn` + `Qwen2.5-1.5B` sweeps; `L4` 24GB or `T4` 16GB work seamlessly with automatic VRAM batch scaling).
2. **Configure Once (Section 2):** Use the Colab `#@param` form widgets to toggle any combination of v2.0 encoders and v3.0 LLM backbones, hyperparameters, RPO-DPO alignment, CFG sweep, and AutoDock Vina docking.
3. **Crash-Proof Execution:** Enable Google Drive sync in Section 1 so every finished checkpoint (`best.pt` / `best_adapter.pt`, `history.jsonl`, `train_summary.json`) is backed up immediately and every stage supports `--resume-skip`.
4. **One-Command Local Integration (Section 11):** Download `geneuclidean_colab_bundle_<timestamp>.tar.gz` and merge weights, tables, and figures into your local Mac/Linux repo with:
   ```bash
   .venv/bin/python scripts/import_colab_bundle.py \
     --bundle ~/Downloads/geneuclidean_colab_bundle_<timestamp>.tar.gz \
     --rebuild-comparison --regenerate-figures
   ```

## Section 1 — Hardware Detection & Google Drive Auto-Sync

Detects your active accelerator (`A100` / `H100` / `L4` / `T4` / `V100` CUDA GPU, Apple MPS, or CPU), inspects VRAM and `bfloat16` support, and optionally mounts Google Drive so checkpoints and benchmark artifacts survive Colab session timeouts.

In [ ]:
#@title 1.1 Hardware Detection & Optional Google Drive Mount
import os
import platform
import shutil
import subprocess
from pathlib import Path

import torch

MOUNT_GOOGLE_DRIVE = True  #@param {type:"boolean"}
DRIVE_SYNC_SUBDIR = "GenEuclidean_v3_Colab"  #@param {type:"string"}

IN_COLAB = False
try:
    import google.colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Inspect accelerator, VRAM, and numerical capabilities
if torch.cuda.is_available():
    ACTIVE_DEVICE = "cuda"
    GPU_NAME = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    VRAM_GIB = props.total_memory / (1024 ** 3)
    SUPPORTS_BF16 = bool(torch.cuda.is_bf16_supported())
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    ACTIVE_DEVICE = "mps"
    GPU_NAME = f"Apple Silicon MPS ({platform.machine()})"
    VRAM_GIB = 24.0
    SUPPORTS_BF16 = True
else:
    ACTIVE_DEVICE = "cpu"
    GPU_NAME = f"CPU ({platform.processor() or platform.machine()})"
    VRAM_GIB = 0.0
    SUPPORTS_BF16 = False

VRAM_TIER = "high" if VRAM_GIB >= 32.0 else ("mid" if VRAM_GIB >= 15.0 else "low")

print("=" * 72)
print(f"Environment   : {'Google Colab' if IN_COLAB else 'Local / Standalone Jupyter'}")
print(f"Python / OS   : {platform.python_version()} on {platform.system()} {platform.machine()}")
print(f"PyTorch       : {torch.__version__} (CUDA build: {torch.version.cuda})")
print(f"Active Device : {ACTIVE_DEVICE} -> {GPU_NAME}")
print(f"VRAM / Tier   : {VRAM_GIB:.1f} GiB (tier={VRAM_TIER}, bfloat16={SUPPORTS_BF16})")
print("=" * 72)

if ACTIVE_DEVICE == "cuda":
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=False)

DRIVE_SYNC_DIR: Path | None = None
if MOUNT_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive  # type: ignore[import-not-found]

    drive.mount("/content/drive", force_remount=False)
    DRIVE_SYNC_DIR = Path("/content/drive/MyDrive") / DRIVE_SYNC_SUBDIR
    DRIVE_SYNC_DIR.mkdir(parents=True, exist_ok=True)
    print(f"[Drive] Crash-proof sync directory ready: {DRIVE_SYNC_DIR}")
elif MOUNT_GOOGLE_DRIVE:
    print("[Drive] Not running inside Google Colab; Drive mount skipped cleanly.")
else:
    print("[Drive] Google Drive sync disabled by user.")


def sync_file_to_drive(src: Path, rel_dst: str) -> None:
    """Copy a single artifact into Google Drive if Drive sync is enabled."""
    if DRIVE_SYNC_DIR is None or not src.is_file():
        return
    dst = DRIVE_SYNC_DIR / rel_dst
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)


def sync_run_dir_to_drive(run_dir: Path, run_rel_name: str) -> None:
    """Immediately back up compact weights and logs after a run finishes."""
    if DRIVE_SYNC_DIR is None or not run_dir.is_dir():
        return
    compact_files = (
        "best.pt",
        "best_adapter.pt",
        "best_prior.pt",
        "config.yaml",
        "history.jsonl",
        "dpo_history.jsonl",
        "train_summary.json",
    )
    copied = 0
    for fname in compact_files:
        fp = run_dir / fname
        if fp.is_file():
            sync_file_to_drive(fp, f"runs/{run_rel_name}/{fname}")
            copied += 1
    if copied:
        print(f"[Drive] Synced {copied} artifacts for runs/{run_rel_name} -> {DRIVE_SYNC_DIR}")

## Section 2 — Master Experiment Configuration (`#@param` Colab Form Widgets)

Configure **every** repository source, benchmark split, v2.0 & v3.0 architecture toggle, training hyperparameter, RPO-DPO alignment setting, and 5-tier evaluation / AutoDock Vina docking control in one place.

> **VRAM Scaling Guide (`BATCH_SIZE = "auto"`):**
> - **A100 / H100 ($\ge 32$ GiB):** `v2=64`, `v3 chemical=32`, `v3 general LLM (0.5B–1.5B)=16 (accum=1)`
> - **L4 / V100 ($15$–$31$ GiB):** `v2=32`, `v3 chemical=16`, `v3 general LLM=8 (accum=2)`
> - **T4 / Low VRAM ($< 15$ GiB):** `v2=32`, `v3 chemical=16`, `v3 general LLM=4 (accum=4)`

In [ ]:
#@title 2.1 Master Experiment & Multi-Architecture Matrix Controls
# --- Repository & Split ---
REPO_URL = "https://github.com/daniil-777/geneuclidean.git"  #@param {type:"string"}
REPO_BRANCH = "main"  #@param {type:"string"}
USE_LOCAL_UPLOAD = False  #@param {type:"boolean"}
SPLIT_TYPE = "casf2016"  #@param ["casf2016", "lp_pdbbind", "scaffold", "random"]
SEED = 42  #@param {type:"integer"}
RESTORE_FROM_DRIVE = True  #@param {type:"boolean"}
RUN_STAGE1_ZINC_PRETRAIN = False  #@param {type:"boolean"}
PRETRAIN_PRIOR_EPOCHS = 6  #@param {type:"integer"}

# --- v2.0 3D Encoder & Representation Matrix ---
TRAIN_V2_PAINN = True  #@param {type:"boolean"}
TRAIN_V2_E3NN = True  #@param {type:"boolean"}
TRAIN_V2_EGNN = False  #@param {type:"boolean"}
TRAIN_V2_SCHNET = False  #@param {type:"boolean"}
TRAIN_V2_SELFIES = False  #@param {type:"boolean"}
TRAIN_V2_POCKETFORMER = False  #@param {type:"boolean"}
TRAIN_V2_E3NN_LSTM_V15 = False  #@param {type:"boolean"}

# --- v3.0 Pretrained Chemical Transformer & Local LLM Matrix ---
TRAIN_V3_GPMOLFORMER = True  #@param {type:"boolean"}
TRAIN_V3_GPT2ZINC = True  #@param {type:"boolean"}
TRAIN_V3_QWEN25_05B = True  #@param {type:"boolean"}
TRAIN_V3_QWEN25_15B = False  #@param {type:"boolean"}
TRAIN_V3_QWEN3_06B = False  #@param {type:"boolean"}
TRAIN_V3_SMOLLM2_360M = False  #@param {type:"boolean"}

# --- Training Hyperparameters ---
V2_EPOCHS = 30  #@param {type:"integer"}
PHASE_A_EPOCHS = 3  #@param {type:"integer"}
PHASE_B_EPOCHS = 12  #@param {type:"integer"}
LLM_PHASE_A_EPOCHS = 1  #@param {type:"integer"}
LLM_PHASE_B_EPOCHS = 4  #@param {type:"integer"}
BATCH_SIZE = "auto"  #@param ["auto", "8", "16", "32", "64"]
LORA_R = 16  #@param {type:"integer"}
LORA_ALPHA = 32  #@param {type:"integer"}
PROJECTOR_LR = 0.001  #@param {type:"number"}
PHASE_B_PROJECTOR_LR = 0.0003  #@param {type:"number"}
LORA_LR = 0.0001  #@param {type:"number"}
NUM_QUERIES = 16  #@param {type:"integer"}
CFG_DROP_PROB = 0.15  #@param {type:"number"}

# --- Stage-3 Stabilized RPO-DPO Alignment ---
RUN_RPO_DPO = True  #@param {type:"boolean"}
DPO_POCKETS = 250  #@param {type:"integer"}
DPO_SAMPLES_PER_POCKET = 16  #@param {type:"integer"}
DPO_BETA = 0.5  #@param {type:"number"}
DPO_SFT_WEIGHT = 0.25  #@param {type:"number"}

# --- 5-Tier Evaluation, CFG Sweep & Native AutoDock Vina Docking ---
EVAL_POCKETS = 100  #@param {type:"integer"}
EVAL_SAMPLES_PER_POCKET = 32  #@param {type:"integer"}
CFG_SCALE = 1.6  #@param {type:"number"}
MIN_P = 0.08  #@param {type:"number"}
TEMPERATURE = 1.0  #@param {type:"number"}
RUN_CFG_SWEEP = True  #@param {type:"boolean"}
RUN_VINA_DOCKING = True  #@param {type:"boolean"}
DOCK_POCKETS = 30  #@param {type:"integer"}


def resolve_batch_sizes(batch_setting: str, vram_tier: str) -> dict[str, int]:
    """Scale batch sizes & gradient accumulation cleanly to the detected GPU VRAM."""
    if batch_setting != "auto":
        bs = max(2, int(batch_setting))
        llm_bs = max(2, bs // 2) if vram_tier != "high" else bs
        llm_accum = max(1, 16 // llm_bs)
        return {"v2": bs, "v3_chem": bs, "v3_llm": llm_bs, "v3_llm_accum": llm_accum}
    if vram_tier == "high":
        return {"v2": 64, "v3_chem": 32, "v3_llm": 16, "v3_llm_accum": 1}
    if vram_tier == "mid":
        return {"v2": 32, "v3_chem": 16, "v3_llm": 8, "v3_llm_accum": 2}
    return {"v2": 32, "v3_chem": 16, "v3_llm": 4, "v3_llm_accum": 4}


RESOLVED_BS = resolve_batch_sizes(BATCH_SIZE, VRAM_TIER)

SELECTED_V2_RUNS = [
    name
    for name, enabled in [
        ("geneuclidean_v2_painn", TRAIN_V2_PAINN),
        ("geneuclidean_v2_e3nn", TRAIN_V2_E3NN),
        ("geneuclidean_v2_egnn", TRAIN_V2_EGNN),
        ("geneuclidean_v2_schnet", TRAIN_V2_SCHNET),
        ("geneuclidean_v2_selfies", TRAIN_V2_SELFIES),
        ("painn_pocketformer", TRAIN_V2_POCKETFORMER),
        ("e3nn_lstm_v15", TRAIN_V2_E3NN_LSTM_V15),
    ]
    if enabled
]

# Declarative v3 LLM matrix: (short_key, cli_model_key, hf_repo_id, is_general_llm, base_yaml)
ALL_V3_CANDIDATES = [
    ("gpmolformer", "gpmolformer", "ibm-research/GP-MoLFormer-Uniq", False, "configs/geneuclidean_v3_gpmolformer.yaml", TRAIN_V3_GPMOLFORMER),
    ("gpt2zinc", "gpt2zinc", "entropy/gpt2_zinc_87m", False, "configs/geneuclidean_v3_gpt2zinc.yaml", TRAIN_V3_GPT2ZINC),
    ("qwen25", "qwen25", "Qwen/Qwen2.5-0.5B", True, "configs/geneuclidean_v3_qwen25.yaml", TRAIN_V3_QWEN25_05B),
    ("qwen25_1p5b", "qwen25", "Qwen/Qwen2.5-1.5B", True, "configs/geneuclidean_v3_qwen25.yaml", TRAIN_V3_QWEN25_15B),
    ("qwen3_0p6b", "qwen25", "Qwen/Qwen3-0.6B-Base", True, "configs/geneuclidean_v3_qwen25.yaml", TRAIN_V3_QWEN3_06B),
    ("smollm2_360m", "qwen25", "HuggingFaceTB/SmolLM2-360M", True, "configs/geneuclidean_v3_qwen25.yaml", TRAIN_V3_SMOLLM2_360M),
]
SELECTED_V3_SPECS = [item[:5] for item in ALL_V3_CANDIDATES if item[5]]

print("Resolved batch sizes :", RESOLVED_BS)
print("Selected v2.0 runs   :", SELECTED_V2_RUNS or "(none)")
print("Selected v3.0 models :", [f"{s[0]} ({s[2]})" for s in SELECTED_V3_SPECS] or "(none)")

## Section 3 — Fast `uv` Environment & Linux AutoDock Vina v1.2.7 Setup

1. Clones `GenEuclidean` (or unpacks an uploaded repository zip if `USE_LOCAL_UPLOAD = True`).
2. Installs `geneuclidean[all]` rapidly via `uv`, **explicitly preserving Colab's CUDA PyTorch** and **pinning `rdkit>=2026.3,<2026.9`** to avoid RDKit `2026.9` Murcko scaffold regressions.
3. Downloads the official Linux `x86_64` **AutoDock Vina v1.2.7** binary (`vina_1.2.7_linux_x86_64`) into `bin/vina` and verifies all installed tooling via `geneuclidean info`.

In [ ]:
#@title 3.1 Clone Repo, Install geneuclidean[all] with uv & Fetch Linux AutoDock Vina v1.2.7
import stat
import urllib.request
import zipfile

WORKDIR = Path("/content/geneuclidean") if IN_COLAB else Path.cwd()

if IN_COLAB:
    if USE_LOCAL_UPLOAD:
        from google.colab import files  # type: ignore[import-not-found]

        print("Upload your local geneuclidean repository .zip archive...")
        uploaded = files.upload()
        zip_name = next(iter(uploaded.keys()))
        WORKDIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_name, "r") as zf:
            zf.extractall(WORKDIR)
        # Unwrap nested single top-level directory if present
        children = [p for p in WORKDIR.iterdir() if p.is_dir() and (p / "pyproject.toml").is_file()]
        if children:
            WORKDIR = children[0]
    elif not (WORKDIR / "pyproject.toml").is_file():
        subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, str(WORKDIR)],
            check=True,
        )
    else:
        print(f"Repository already present at {WORKDIR}; pulling latest {REPO_BRANCH}...")
        subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only"], check=False)

    os.chdir(WORKDIR)

REPO_ROOT = Path.cwd()
HF_CACHE_DIR = REPO_ROOT / ".hf_cache.nosync"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR.resolve())
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Fast uv installation (preserves preinstalled CUDA torch on Colab!)
if IN_COLAB:
    subprocess.run([os.sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    # Install package + [llm,docking,dev] extras without clobbering Colab's CUDA torch
    subprocess.run(
        [
            "uv",
            "pip",
            "install",
            "--system",
            "--quiet",
            "rdkit>=2026.3,<2026.9",
            "e3nn>=0.5",
            "selfies>=2.1",
            "biopython>=1.83",
            "meeko>=0.8",
            "gemmi>=0.7.5",
            "transformers>=5.0",
            "peft>=0.15",
            "safetensors>=0.4",
            "accelerate>=1.0",
            "huggingface-hub>=1.0",
            "typer>=0.12",
            "rich>=13.7",
            "pydantic>=2.0",
            "pytest>=8.0",
            "pytest-cov>=5.0",
        ],
        check=True,
    )
    subprocess.run(
        ["uv", "pip", "install", "--system", "--quiet", "--no-deps", "-e", "."],
        check=True,
    )

# Install native AutoDock Vina v1.2.7 binary for Linux x86_64 (Colab)
VINA_BIN = REPO_ROOT / "bin" / "vina"
if not VINA_BIN.is_file():
    VINA_BIN.parent.mkdir(parents=True, exist_ok=True)
    if platform.system() == "Linux" and platform.machine() in ("x86_64", "amd64"):
        vina_url = (
            "https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/"
            "v1.2.7/vina_1.2.7_linux_x86_64"
        )
        print(f"Downloading Linux AutoDock Vina v1.2.7 -> {VINA_BIN}...")
        urllib.request.urlretrieve(vina_url, str(VINA_BIN))
        VINA_BIN.chmod(VINA_BIN.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    else:
        print(f"Non-Linux/x86_64 host ({platform.system()} {platform.machine()}); using repo downloader.")
        subprocess.run([os.sys.executable, "-m", "geneuclidean.cli", "download"], check=False)

if VINA_BIN.is_file():
    ver = subprocess.run([str(VINA_BIN), "--version"], capture_output=True, text=True, check=False)
    print("AutoDock Vina :", (ver.stdout or ver.stderr).strip())

subprocess.run([os.sys.executable, "-m", "geneuclidean.cli", "info"], check=True)

In [ ]:
#@title 3.2 Generate Validated v3 YAML Overrides & Preview Experiment Plan
import yaml

from geneuclidean.config import ExperimentConfig, LLMConfig

COLAB_CFG_DIR = REPO_ROOT / "configs" / "colab_generated"
COLAB_CFG_DIR.mkdir(parents=True, exist_ok=True)

GENERATED_V3_CONFIGS: dict[str, Path] = {}
V3_SPLIT_SUFFIX = "" if SPLIT_TYPE == "casf2016" else ("_lp" if SPLIT_TYPE == "lp_pdbbind" else f"_{SPLIT_TYPE}")
V2_PAINN_CKPT = REPO_ROOT / "runs" / f"geneuclidean_v2_painn{V3_SPLIT_SUFFIX}" / "best.pt"
if not V2_PAINN_CKPT.is_file():
    V2_PAINN_CKPT = REPO_ROOT / "runs" / "geneuclidean_v2_painn" / "best.pt"

for short_key, _cli_key, backbone_id, is_general_llm, base_yaml in SELECTED_V3_SPECS:
    cfg = ExperimentConfig.from_yaml(REPO_ROOT / base_yaml)
    llm_cfg = cfg.llm if cfg.llm is not None else LLMConfig()

    cfg.name = f"geneuclidean-v3-{short_key}"
    cfg.data.split_type = SPLIT_TYPE
    cfg.data.randomize_smiles = not is_general_llm
    cfg.train.seed = int(SEED)
    cfg.train.device = ACTIVE_DEVICE
    cfg.train.batch_size = RESOLVED_BS["v3_llm"] if is_general_llm else RESOLVED_BS["v3_chem"]
    cfg.train.output_dir = f"runs/geneuclidean_v3_{short_key}{V3_SPLIT_SUFFIX}"

    cfg.sampling.num_samples = int(EVAL_SAMPLES_PER_POCKET)
    cfg.sampling.cfg_scale = float(CFG_SCALE)
    cfg.sampling.min_p = float(MIN_P)
    cfg.sampling.temperature = float(TEMPERATURE)

    llm_cfg.backbone_id = backbone_id
    llm_cfg.cache_dir = str(HF_CACHE_DIR)
    llm_cfg.dtype = "bfloat16" if (is_general_llm and SUPPORTS_BF16) else "float32"
    llm_cfg.v2_pocket_ckpt = str(V2_PAINN_CKPT)
    llm_cfg.num_queries = int(NUM_QUERIES)
    llm_cfg.lora_r = int(LORA_R)
    llm_cfg.lora_alpha = int(LORA_ALPHA)
    llm_cfg.phase_a_epochs = int(LLM_PHASE_A_EPOCHS if is_general_llm else PHASE_A_EPOCHS)
    llm_cfg.phase_b_epochs = int(LLM_PHASE_B_EPOCHS if is_general_llm else PHASE_B_EPOCHS)
    llm_cfg.projector_lr = float(PROJECTOR_LR)
    llm_cfg.phase_b_projector_lr = float(PHASE_B_PROJECTOR_LR)
    llm_cfg.lora_lr = float(LORA_LR)
    llm_cfg.cfg_drop_prob = float(CFG_DROP_PROB)
    llm_cfg.grad_accum_steps = int(RESOLVED_BS["v3_llm_accum"] if is_general_llm else 1)
    cfg.llm = llm_cfg

    out_yaml = COLAB_CFG_DIR / f"geneuclidean_v3_{short_key}.yaml"
    cfg.to_yaml(out_yaml)
    GENERATED_V3_CONFIGS[short_key] = out_yaml

if GENERATED_V3_CONFIGS:
    first_key = next(iter(GENERATED_V3_CONFIGS))
    print(f"--- Auto-generated validated YAML preview ({GENERATED_V3_CONFIGS[first_key].name}) ---")
    preview_data = yaml.safe_load(GENERATED_V3_CONFIGS[first_key].read_text(encoding="utf-8"))
    print(yaml.safe_dump({"name": preview_data["name"], "train": preview_data["train"], "llm": preview_data["llm"]}, sort_keys=False))

## Section 4 — Automated Data Download & Zero-Leakage Preprocessing

Streams the compact 254 MB `pbpp-2020` protein-pocket/ligand structures, `LP_PDBBind.csv`, `CASF-2016` core set, and `ZINC-250k`, builds `data/processed/pdbbind_pockets.pt` and `data/processed/splits.json`, and programmatically asserts **exact `0 / 285` CASF-2016 train/val leakage**.

In [ ]:
#@title 4.1 Download Datasets, Build Pocket Cache & Assert Zero CASF-2016 Leakage
import json

CACHE_PATH = REPO_ROOT / "data" / "processed" / "pdbbind_pockets.pt"
SPLITS_PATH = REPO_ROOT / "data" / "processed" / "splits.json"

if not (CACHE_PATH.is_file() and SPLITS_PATH.is_file()):
    subprocess.run(
        [os.sys.executable, "-m", "geneuclidean.cli", "download", "--no-vina"],
        check=True,
    )
    subprocess.run(
        [os.sys.executable, "-m", "geneuclidean.cli", "prepare", "--seed", str(SEED)],
        check=True,
    )
else:
    print(f"[Data] Processed cache already present: {CACHE_PATH} ({CACHE_PATH.stat().st_size / 1e6:.1f} MB)")

splits_data = json.loads(SPLITS_PATH.read_text(encoding="utf-8"))
casf_core = set(splits_data["casf2016_core_ids"])
casf_train = set(splits_data["casf2016"]["train"])
casf_val = set(splits_data["casf2016"]["val"])
casf_test = set(splits_data["casf2016"]["test"])

train_leak = casf_core & casf_train
val_leak = casf_core & casf_val
assert len(train_leak) == 0, f"CRITICAL LEAKAGE in train: {train_leak}"
assert len(val_leak) == 0, f"CRITICAL LEAKAGE in val: {val_leak}"

print(
    f"[Zero-Leakage Audit PASS] CASF-2016 Core={len(casf_core)} | "
    f"Train={len(casf_train)} (leak=0) | Val={len(casf_val)} (leak=0) | Test={len(casf_test)}"
)
print(
    f"[LP-PDBBind Audit] Train={len(splits_data['lp_pdbbind']['train'])} | "
    f"Val={len(splits_data['lp_pdbbind']['val'])} | Test={len(splits_data['lp_pdbbind']['test'])}"
)

## Section 5 — Mathematical $SO(3)/SE(3)$/Permutation Equivariance & Pytest Sanity Gate

Before spending GPU minutes on training, we verify mathematically on the live runtime that:
- `PaiNN-SE3`, `EGNN`, `e3nn`, and `SchNet` satisfy exact `$<10^{-5}$` $SO(3)$ rotation, $\mathbb{R}^3$ translation, and atom-permutation invariance (with chiral sensitivity for `PaiNN-SE3` & `e3nn`).
- `PocketPerceiverResampler` and `PocketConditionedLLM` maintain exact `0.0` attention weight on masked dummy atoms, `$<10^{-5}$` $SE(3)$ prefix invariance, and exact KV-cached CFG logit parity.

In [ ]:
#@title 5.1 Run Live Symmetry, Perceiver Bridge & KV-Cache Pytest Gate
RUN_SANITY_TESTS = True  #@param {type:"boolean"}

if RUN_SANITY_TESTS:
    subprocess.run(
        [
            os.sys.executable,
            "-m",
            "pytest",
            "-q",
            "-m",
            "not slow",
            "tests/test_equivariance.py",
            "tests/test_llm_bridge.py",
            "tests/test_llm_trainer.py",
        ],
        check=True,
    )
    print("[Sanity Gate PASS] All SE(3)/E(3) equivariance, Perceiver bridge, and trainer tests passed!")
else:
    print("[Sanity Gate] Skipped by user.")

## Section 6 — Optional Stage-1 Chemical Prior Pretraining (ZINC-250k) or Drive Restore

Restores previously saved `runs/` artifacts from Google Drive (if enabled) so you can warm-start v3.0 LLMs directly from an uploaded `runs/geneuclidean_v2_painn/best.pt`, or pretrains Stage-1 ZINC-250k SMILES / SELFIES chemical priors when training v2.0 models from scratch.

In [ ]:
#@title 6.1 Restore Existing Checkpoints from Google Drive & Optional Stage-1 Prior Pretraining
if RESTORE_FROM_DRIVE and DRIVE_SYNC_DIR is not None and (DRIVE_SYNC_DIR / "runs").is_dir():
    restored = 0
    for src_file in (DRIVE_SYNC_DIR / "runs").rglob("*"):
        if not src_file.is_file():
            continue
        rel = src_file.relative_to(DRIVE_SYNC_DIR)
        dst = REPO_ROOT / rel
        if not dst.is_file():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_file, dst)
            restored += 1
    print(f"[Drive Restore] Restored {restored} cached checkpoint/log files into {REPO_ROOT / 'runs'}.")

# Train Stage-1 ZINC-250k chemical priors if requested or required by selected v2 runs
need_smiles_prior = RUN_STAGE1_ZINC_PRETRAIN or any(
    r in SELECTED_V2_RUNS
    for r in (
        "geneuclidean_v2_painn",
        "geneuclidean_v2_e3nn",
        "geneuclidean_v2_egnn",
        "geneuclidean_v2_schnet",
        "painn_pocketformer",
    )
)
need_selfies_prior = "geneuclidean_v2_selfies" in SELECTED_V2_RUNS

smiles_prior_ckpt = REPO_ROOT / "runs" / "prior_smiles" / "best_prior.pt"
if need_smiles_prior and not smiles_prior_ckpt.is_file():
    print(f"[Stage-1] Pretraining ZINC-250k SMILES chemical prior ({PRETRAIN_PRIOR_EPOCHS} epochs)...")
    subprocess.run(
        [
            os.sys.executable,
            "scripts/run_pretrain.py",
            "--config",
            "configs/pretrain_zinc.yaml",
            "--output-dir",
            "runs/prior_smiles",
            "--epochs",
            str(PRETRAIN_PRIOR_EPOCHS),
            "--batch-size",
            "128",
            "--seed",
            str(SEED),
        ],
        check=True,
    )
    sync_run_dir_to_drive(REPO_ROOT / "runs" / "prior_smiles", "prior_smiles")
elif smiles_prior_ckpt.is_file():
    print(f"[Stage-1] SMILES prior ready: {smiles_prior_ckpt}")

selfies_prior_ckpt = REPO_ROOT / "runs" / "prior_selfies" / "best_prior.pt"
if need_selfies_prior and not selfies_prior_ckpt.is_file():
    print(f"[Stage-1] Pretraining ZINC-250k SELFIES chemical prior ({PRETRAIN_PRIOR_EPOCHS} epochs)...")
    subprocess.run(
        [
            os.sys.executable,
            "scripts/run_pretrain.py",
            "--config",
            "configs/pretrain_zinc_selfies.yaml",
            "--output-dir",
            "runs/prior_selfies",
            "--epochs",
            str(PRETRAIN_PRIOR_EPOCHS),
            "--batch-size",
            "128",
            "--seed",
            str(SEED),
        ],
        check=True,
    )
    sync_run_dir_to_drive(REPO_ROOT / "runs" / "prior_selfies", "prior_selfies")
elif selfies_prior_ckpt.is_file():
    print(f"[Stage-1] SELFIES prior ready: {selfies_prior_ckpt}")

## Section 7 — Automated Multi-Architecture Training Loop (v2.0 + v3.0 LLMs)

Iterates cleanly across every selected v2.0 3D architecture (`PaiNN-SE3`, `e3nn`, `EGNN`, `SchNet`, `SELFIES`, `PaiNN-PocketFormer`, `v1.5 e3nn+LSTM`) and every selected v3.0 Pretrained Chemical Foundation Transformer / Local LLM (`GP-MoLFormer-Uniq`, `GPT2-ZINC-87M`, `Qwen2.5-0.5B/1.5B`, `Qwen3-0.6B-Base`, `SmolLM2-360M`).

- Every run uses `--resume-skip` so re-running after a disconnect never repeats completed models.
- Every finished checkpoint is immediately synced to Google Drive!

In [ ]:
#@title 7.1 Train Selected v2.0 3D Models (PaiNN-SE3, e3nn, EGNN, SchNet, SELFIES, ...)
for run_name in SELECTED_V2_RUNS:
    print(f"\n>>> [v2.0] Starting {run_name} (split={SPLIT_TYPE}, epochs={V2_EPOCHS}, bs={RESOLVED_BS['v2']})...")
    subprocess.run(
        [
            os.sys.executable,
            "scripts/run_stage2_suite.py",
            "--run",
            run_name,
            "--split",
            SPLIT_TYPE,
            "--epochs",
            str(V2_EPOCHS),
            "--batch-size",
            str(RESOLVED_BS["v2"]),
            "--device",
            ACTIVE_DEVICE,
            "--seed",
            str(SEED),
            "--resume-skip",
        ],
        check=True,
    )
    run_folder = f"{run_name}{V3_SPLIT_SUFFIX}"
    sync_run_dir_to_drive(REPO_ROOT / "runs" / run_folder, run_folder)

# Refresh resolved v2 PaiNN warm-start checkpoint path for v3 LLMs
for cand_ckpt in (
    REPO_ROOT / "runs" / f"geneuclidean_v2_painn{V3_SPLIT_SUFFIX}" / "best.pt",
    REPO_ROOT / "runs" / "geneuclidean_v2_painn" / "best.pt",
):
    if cand_ckpt.is_file():
        V2_PAINN_CKPT = cand_ckpt
        break

print(f"\n[v2.0 Complete] Active warm-start PaiNN-SE3 checkpoint: {V2_PAINN_CKPT} (exists={V2_PAINN_CKPT.is_file()})")

In [ ]:
#@title 7.2 Train Selected v3.0 Pretrained Chemical Transformers & Local LLMs (2-Phase Perceiver + LoRA)
if SELECTED_V3_SPECS and not V2_PAINN_CKPT.is_file():
    raise FileNotFoundError(
        f"v2.0 PaiNN-SE3 checkpoint not found at {V2_PAINN_CKPT}. "
        "Enable TRAIN_V2_PAINN = True in Section 2 or upload runs/geneuclidean_v2_painn/best.pt to Drive."
    )

for short_key, cli_model_key, backbone_id, is_general_llm, _base_yaml in SELECTED_V3_SPECS:
    cfg_yaml = GENERATED_V3_CONFIGS[short_key]
    run_folder = f"geneuclidean_v3_{short_key}{V3_SPLIT_SUFFIX}"
    out_dir = REPO_ROOT / "runs" / run_folder
    ph_a = LLM_PHASE_A_EPOCHS if is_general_llm else PHASE_A_EPOCHS
    ph_b = LLM_PHASE_B_EPOCHS if is_general_llm else PHASE_B_EPOCHS
    bs = RESOLVED_BS["v3_llm"] if is_general_llm else RESOLVED_BS["v3_chem"]

    print(
        f"\n>>> [v3.0] Training {short_key} ({backbone_id}) | "
        f"Phase A={ph_a} ep + Phase B={ph_b} ep | bs={bs} | device={ACTIVE_DEVICE}"
    )
    subprocess.run(
        [
            os.sys.executable,
            "scripts/run_v3_llm.py",
            "--model",
            cli_model_key,
            "--config",
            str(cfg_yaml),
            "--v2-ckpt",
            str(V2_PAINN_CKPT),
            "--output-dir",
            str(out_dir),
            "--split",
            SPLIT_TYPE,
            "--phase-a-epochs",
            str(ph_a),
            "--phase-b-epochs",
            str(ph_b),
            "--batch-size",
            str(bs),
            "--cache-dir",
            str(HF_CACHE_DIR),
            "--device",
            ACTIVE_DEVICE,
            "--seed",
            str(SEED),
            "--resume-skip",
        ],
        check=True,
    )
    sync_run_dir_to_drive(out_dir, run_folder)

## Section 8 — Automated Stage-3 Stabilized RPO-DPO Alignment (v2.0 + v3.0)

Runs length-normalized **RPO-DPO** (`sft_nll_weight = 0.25`, `grad_accum_steps = 8`) over on-policy MPO-scored preference pairs (combining predicted `pKd`, `DrugCLIP` similarity, `QED`, `SA`, `Lipinski`, and `PAINS`-free indicators) while **freezing `PaiNN-SE3`, `CVAE`, `Perceiver`, `pKd`, and `DrugCLIP` bit-for-bit**!

In [ ]:
#@title 8.1 Run Stage-3 RPO-DPO Alignment on Flagship v2.0 & v3.0 Models
if RUN_RPO_DPO:
    # 1. Optional v2.0 PaiNN-SE3 RPO-DPO
    if V2_PAINN_CKPT.is_file() and "geneuclidean_v2_painn" in SELECTED_V2_RUNS:
        print("\n>>> [Stage-3 DPO] Aligning v2.0 Flagship PaiNN-SE3...")
        subprocess.run(
            [
                os.sys.executable,
                "scripts/run_stage2_suite.py",
                "--run",
                "geneuclidean_v2_painn_dpo",
                "--split",
                SPLIT_TYPE,
                "--init-ckpt",
                str(V2_PAINN_CKPT),
                "--dpo-pockets",
                str(DPO_POCKETS),
                "--dpo-samples-per-pocket",
                str(DPO_SAMPLES_PER_POCKET),
                "--dpo-beta",
                str(DPO_BETA),
                "--dpo-sft-weight",
                str(DPO_SFT_WEIGHT),
                "--device",
                ACTIVE_DEVICE,
                "--seed",
                str(SEED),
                "--resume-skip",
            ],
            check=True,
        )
        dpo_v2_folder = f"geneuclidean_v2_painn_dpo{V3_SPLIT_SUFFIX}"
        sync_run_dir_to_drive(REPO_ROOT / "runs" / dpo_v2_folder, dpo_v2_folder)

    # 2. v3.0 LoRA-Only RPO-DPO for flagship chemical foundation model(s)
    for dpo_base in ("gpmolformer", "gpt2zinc"):
        parent_adapter = REPO_ROOT / "runs" / f"geneuclidean_v3_{dpo_base}{V3_SPLIT_SUFFIX}" / "best_adapter.pt"
        if not parent_adapter.is_file():
            continue
        dpo_folder = f"geneuclidean_v3_{dpo_base}_dpo{V3_SPLIT_SUFFIX}"
        dpo_out = REPO_ROOT / "runs" / dpo_folder
        cfg_yaml = GENERATED_V3_CONFIGS.get(dpo_base, REPO_ROOT / f"configs/geneuclidean_v3_{dpo_base}.yaml")
        print(f"\n>>> [Stage-3 v3 LoRA RPO-DPO] Aligning {dpo_base} -> {dpo_folder}...")
        subprocess.run(
            [
                os.sys.executable,
                "scripts/run_v3_llm.py",
                "--model",
                f"{dpo_base}_dpo",
                "--config",
                str(cfg_yaml),
                "--v2-ckpt",
                str(V2_PAINN_CKPT),
                "--init-adapter",
                str(parent_adapter),
                "--output-dir",
                str(dpo_out),
                "--split",
                SPLIT_TYPE,
                "--dpo-pockets",
                str(DPO_POCKETS),
                "--dpo-samples-per-pocket",
                str(DPO_SAMPLES_PER_POCKET),
                "--dpo-beta",
                str(DPO_BETA),
                "--dpo-sft-weight",
                str(DPO_SFT_WEIGHT),
                "--cache-dir",
                str(HF_CACHE_DIR),
                "--device",
                ACTIVE_DEVICE,
                "--seed",
                str(SEED),
                "--resume-skip",
            ],
            check=True,
        )
        sync_run_dir_to_drive(dpo_out, dpo_folder)
else:
    print("[Stage-3 DPO] Skipped (RUN_RPO_DPO = False).")

## Section 9 — Automated Unified 5-Tier CASF-2016 Benchmark, Decoding/CFG Pareto Sweep & Native AutoDock Vina v1.2.7 + PoseBusters Docking

Evaluates all trained v2.0 & v3.0 models under the **exact same 5-tier zero-leakage protocol**:
- **Tier 1:** Validity, Uniqueness, Novelty, `IntDiv1/2`, Murcko Scaffold Diversity, QED, SA, Lipinski, PAINS-free, FFD, W1, JSD (raw & MPO+MaxMin Top-10).
- **Tier 2:** Causal 3D pocket conditioning — Matched vs. Shuffled Derangement vs. Null Pocket (`True Tanimoto`, `Count Dice`, Wilcoxon, Mann-Whitney U, Cliff's $\delta$, MW/LogP/TPSA Pearson $r$).
- **Tier 3:** `<4.5 Å` Contact Atom Attention Enrichment & residue-level heatmaps.
- **Tier 4:** Full 285-complex CASF-2016 `pKd` scoring power & `DrugCLIP` retrieval.
- **Tier 5:** RDKit `ETKDGv3 + MMFF94s` conformer generation + **AutoDock Vina v1.2.7** docking + **PoseBusters** physical validity checks!

In [ ]:
#@title 9.1 Run v2.0 & v3.0 5-Tier CASF-2016 Evaluations, CFG Sweep & Vina Docking
TABLES_DIR = REPO_ROOT / "results" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# 1. Evaluate all trained v2.0 checkpoints + 15-config decoding Pareto sweep
v2_ckpts = sorted((REPO_ROOT / "runs").glob("*/best.pt"))
if v2_ckpts:
    print(f">>> [Benchmark v2.0] Evaluating {len(v2_ckpts)} trained v2 checkpoints...")
    subprocess.run(
        [
            os.sys.executable,
            "scripts/run_benchmark.py",
            "--mode",
            "models",
            "--split",
            SPLIT_TYPE,
            "--max-pockets",
            str(EVAL_POCKETS),
            "--num-samples",
            str(EVAL_SAMPLES_PER_POCKET),
            "--device",
            ACTIVE_DEVICE,
            "--seed",
            str(SEED),
            "--resume-skip",
        ],
        check=True,
    )
    if RUN_CFG_SWEEP and V2_PAINN_CKPT.is_file():
        subprocess.run(
            [
                os.sys.executable,
                "scripts/run_benchmark.py",
                "--mode",
                "decoding",
                "--checkpoint",
                str(V2_PAINN_CKPT),
                "--split",
                SPLIT_TYPE,
                "--decoding-pockets",
                str(min(50, EVAL_POCKETS)),
                "--num-samples",
                str(EVAL_SAMPLES_PER_POCKET),
                "--device",
                ACTIVE_DEVICE,
                "--seed",
                str(SEED),
                "--resume-skip",
            ],
            check=True,
        )

# 2. Evaluate all trained v3.0 LLM adapters + optional CFG/Hit-to-Lead sweep
v3_adapters = sorted((REPO_ROOT / "runs").glob("geneuclidean_v3_*/best_adapter.pt"))
if v3_adapters:
    print(f">>> [Benchmark v3.0] Evaluating {len(v3_adapters)} trained v3 LLM adapters...")
    eval_cmd = [
        os.sys.executable,
        "scripts/eval_v3_llm.py",
        "--split",
        SPLIT_TYPE,
        "--max-pockets",
        str(EVAL_POCKETS),
        "--num-samples",
        str(EVAL_SAMPLES_PER_POCKET),
        "--cfg-scale",
        str(CFG_SCALE),
        "--min-p",
        str(MIN_P),
        "--temperature",
        str(TEMPERATURE),
        "--device",
        ACTIVE_DEVICE,
        "--seed",
        str(SEED),
        "--resume-skip",
    ]
    if RUN_CFG_SWEEP:
        eval_cmd.extend(["--cfg-sweep", "--sweep-pockets", str(min(50, EVAL_POCKETS))])
    subprocess.run(eval_cmd, check=True)

# 3. Native Linux AutoDock Vina v1.2.7 + PoseBusters 3D Docking
if RUN_VINA_DOCKING and VINA_BIN.is_file():
    dock_candidates = [
        TABLES_DIR / "samples_v3_gpmolformer_dpo.json",
        TABLES_DIR / "samples_v3_gpmolformer.json",
        TABLES_DIR / "samples_v3_gpt2zinc_dpo.json",
        TABLES_DIR / "samples_v3_gpt2zinc.json",
        TABLES_DIR / "samples_geneuclidean_v2_painn_dpo.json",
        TABLES_DIR / "samples_geneuclidean_v2_painn.json",
    ]
    dock_samples = next((p for p in dock_candidates if p.is_file()), None)
    if dock_samples is not None:
        n_workers = max(2, min(8, (os.cpu_count() or 2) // 2))
        print(f">>> [Tier-5 Docking] Running AutoDock Vina v1.2.7 on {dock_samples.name} ({DOCK_POCKETS} pockets, workers={n_workers})...")
        subprocess.run(
            [
                os.sys.executable,
                "scripts/run_benchmark.py",
                "--mode",
                "docking",
                "--samples-file",
                str(dock_samples),
                "--split",
                SPLIT_TYPE,
                "--dock-pockets",
                str(DOCK_POCKETS),
                "--dock-top-k",
                "5",
                "--dock-exhaustiveness",
                "4",
                "--dock-workers",
                str(n_workers),
                "--vina-bin",
                str(VINA_BIN),
                "--seed",
                str(SEED),
                "--resume-skip",
            ],
            check=True,
        )
    else:
        print("[Tier-5 Docking] No samples_*.json found yet; skipping docking.")
else:
    print("[Tier-5 Docking] Skipped (disabled or Vina binary unavailable).")

## Section 10 — Interactive Publication Visualization Gallery

Renders all 300-DPI publication figures (`scripts/generate_figures.py`) and displays both the unified comparison table (`2021 Thesis` + `GenEuclidean v2.0` + `GenEuclidean v3.0`) and every rendered PNG figure inline.

In [ ]:
#@title 10.1 Render 300-DPI Figures & Display Interactive Comparison Table + Gallery
import pandas as pd
from IPython.display import Image, Markdown, display

FIGURES_DIR = REPO_ROOT / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        os.sys.executable,
        "scripts/generate_figures.py",
        "--runs-dir",
        str(REPO_ROOT / "runs"),
        "--tables-dir",
        str(TABLES_DIR),
        "--output-dir",
        str(FIGURES_DIR),
        "--split",
        SPLIT_TYPE,
    ],
    check=False,
)

# Display unified comparison table (v3 unified table if available, else v2 table)
comp_csv = TABLES_DIR / "model_comparison_v3.csv"
if not comp_csv.is_file():
    comp_csv = TABLES_DIR / "model_comparison.csv"

if comp_csv.is_file():
    df_cmp = pd.read_csv(comp_csv)
    display(Markdown(f"### Unified Benchmark Comparison (`{comp_csv.name}`)"))
    display(df_cmp)
else:
    print("No comparison CSV generated yet.")

# Render every generated 300-DPI figure inline
png_files = sorted(FIGURES_DIR.glob("*.png"))
print(f"Rendering {len(png_files)} publication figures from {FIGURES_DIR}...")
for png in png_files:
    display(Markdown(f"#### `{png.name}`"))
    display(Image(filename=str(png), width=900))

## Section 11 — One-Click Export & Download Bundle (`geneuclidean_colab_bundle_<timestamp>.tar.gz`)

Packages **strictly compact, verified artifacts** into a single self-contained `.tar.gz` bundle with a SHA256-verified `manifest.json`:
- Trained compact model checkpoints (`best.pt`, `best_adapter.pt`, `best_prior.pt`) + `config.yaml`, `history.jsonl`, `dpo_history.jsonl`, `train_summary.json` (skipping bulky temporary optimizer states / `last.pt`).
- All benchmark tables & sample JSONs (`results/tables/*`).
- All 300-DPI publication figures (`results/figures/*.png`).
- `manifest.json` recording GPU hardware, git commit, hyperparameters, headline metrics, and SHA256 checksums for every file.

After downloading, integrate everything cleanly into your local repo in **one command**:
```bash
.venv/bin/python scripts/import_colab_bundle.py \
  --bundle ~/Downloads/geneuclidean_colab_bundle_<timestamp>.tar.gz \
  --rebuild-comparison --regenerate-figures
```

In [ ]:
#@title 11.1 Build SHA256-Verified Export Bundle, Sync to Drive & Trigger Download
import datetime
import hashlib
import io
import tarfile

TIMESTAMP = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
BUNDLE_NAME = f"geneuclidean_colab_bundle_{TIMESTAMP}.tar.gz"
BUNDLE_PATH = REPO_ROOT / BUNDLE_NAME

COMPACT_RUN_FILENAMES = {
    "best.pt",
    "best_adapter.pt",
    "best_prior.pt",
    "config.yaml",
    "history.jsonl",
    "dpo_history.jsonl",
    "train_summary.json",
}

def _sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# Collect all compact files to package with strict relative paths under runs/ and results/
files_to_pack: list[tuple[Path, str]] = []

runs_root = REPO_ROOT / "runs"
if runs_root.is_dir():
    for run_dir in sorted(runs_root.iterdir()):
        if not run_dir.is_dir():
            continue
        for fname in sorted(COMPACT_RUN_FILENAMES):
            fp = run_dir / fname
            if fp.is_file():
                files_to_pack.append((fp, f"runs/{run_dir.name}/{fname}"))

if TABLES_DIR.is_dir():
    for fp in sorted(TABLES_DIR.rglob("*")):
        if fp.is_file():
            rel = fp.relative_to(REPO_ROOT).as_posix()
            files_to_pack.append((fp, rel))

if FIGURES_DIR.is_dir():
    for fp in sorted(FIGURES_DIR.glob("*.png")):
        if fp.is_file():
            rel = fp.relative_to(REPO_ROOT).as_posix()
            files_to_pack.append((fp, rel))

git_rev = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
    check=False,
).stdout.strip() or "unknown"

headline_metrics: list[dict[str, Any]] = []
if comp_csv.is_file():
    headline_metrics = pd.read_csv(comp_csv).where(pd.notna, None).to_dict(orient="records")

manifest_files: list[dict[str, Any]] = []
for src_path, arc_rel in files_to_pack:
    manifest_files.append(
        {
            "path": arc_rel,
            "size_bytes": int(src_path.stat().st_size),
            "sha256": _sha256_file(src_path),
        }
    )

manifest_payload = {
    "bundle_version": "3.0.0",
    "created_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "git_commit": git_rev,
    "gpu_name": GPU_NAME,
    "vram_gib": round(float(VRAM_GIB), 2),
    "active_device": ACTIVE_DEVICE,
    "split_type": SPLIT_TYPE,
    "seed": int(SEED),
    "hyperparameters": {
        "v2_epochs": int(V2_EPOCHS),
        "phase_a_epochs": int(PHASE_A_EPOCHS),
        "phase_b_epochs": int(PHASE_B_EPOCHS),
        "lora_r": int(LORA_R),
        "lora_alpha": int(LORA_ALPHA),
        "projector_lr": float(PROJECTOR_LR),
        "phase_b_projector_lr": float(PHASE_B_PROJECTOR_LR),
        "lora_lr": float(LORA_LR),
        "num_queries": int(NUM_QUERIES),
        "cfg_drop_prob": float(CFG_DROP_PROB),
        "cfg_scale": float(CFG_SCALE),
        "min_p": float(MIN_P),
        "temperature": float(TEMPERATURE),
        "resolved_batch_sizes": RESOLVED_BS,
    },
    "headline_metrics": headline_metrics,
    "files": manifest_files,
}

manifest_bytes = (json.dumps(manifest_payload, indent=2) + "\n").encode("utf-8")

with tarfile.open(BUNDLE_PATH, mode="w:gz") as tf:
    info = tarfile.TarInfo(name="manifest.json")
    info.size = len(manifest_bytes)
    tf.addfile(info, io.BytesIO(manifest_bytes))
    for src_path, arc_rel in files_to_pack:
        tf.add(src_path, arcname=arc_rel, recursive=False)

bundle_mb = BUNDLE_PATH.stat().st_size / (1024.0 * 1024.0)
print(f"[Bundle Ready] {BUNDLE_PATH} ({bundle_mb:.2f} MB, {len(manifest_files)} verified files)")

if DRIVE_SYNC_DIR is not None:
    drive_bundle = DRIVE_SYNC_DIR / BUNDLE_NAME
    shutil.copy2(BUNDLE_PATH, drive_bundle)
    print(f"[Drive Backup] Copied bundle -> {drive_bundle}")

print("\nRun this command locally on your Mac/Linux workstation to import:")
print(
    f"  .venv/bin/python scripts/import_colab_bundle.py "
    f"--bundle ~/Downloads/{BUNDLE_NAME} --rebuild-comparison --regenerate-figures"
)

if IN_COLAB:
    from google.colab import files  # type: ignore[import-not-found]

    files.download(str(BUNDLE_PATH))